# nuReasoning reasoning tutorial

This tutorial walks through the reasoning-only benchmark: turning reasoning annotations into
question-answer pairs, LoRA fine-tuning a vision-language model on them, and scoring its answers.
The planning action expert is not involved; for the joint vision-language-action model see
`nureasoning_planning_tutorial.ipynb`.

Prerequisites: the devkit environment (`pip install -e .`), a CUDA GPU, and the train and
validation splits downloaded (see `nureasoning_data_visualization.ipynb`). The first code cell
`chdir`s to the repository root so paths match the README and `docs/reasoning.md`.

Full reference: [`docs/reasoning.md`](../docs/reasoning.md).

## 1. Generate question-answer pairs

Both training and evaluation start from VQA files, one `<timestamp>_vqa.json` per annotated frame.
Generate them once per split. Each file also carries a `temporal_multiview_context` block that
indexes the 8 camera images at the current frame and at the preceding 1 Hz steps (`--history-frames`,
default 3). The question text does not depend on that window; `build_sft` and `evaluate` later keep
only the most recent `--num-forward-frames` frames per camera (2 = current plus 1 s earlier).

In [ ]:
import os
from pathlib import Path

_here = Path.cwd()
REPO_ROOT = _here if (_here / "nureasoning").is_dir() else _here.parent
os.chdir(REPO_ROOT)

TRAIN_ROOT = Path(os.environ.get("NUREASONING_TRAIN_ROOT", "./dataset/data/train"))
VALIDATION_ROOT = Path(os.environ.get("NUREASONING_VALIDATION_ROOT", "./dataset/data/validation"))
VQA_TRAIN_DIR = Path(os.environ.get("NUREASONING_VQA_TRAIN", "./vqa_output_train"))
VQA_VAL_DIR = Path(os.environ.get("NUREASONING_VQA_VAL", "./vqa_output_val"))
REASONING_WORKSPACE = Path(os.environ.get("NUREASONING_REASONING_WORKSPACE", "./reasoning_workspace"))
REASONING_EVAL_ROOT = Path(os.environ.get("NUREASONING_REASONING_EVAL", "./reasoning_eval"))
REASONING_CONFIG = Path(os.environ.get(
    "NUREASONING_REASONING_CONFIG",
    "nureasoning/reasoning/configs/qwen3.5-4b-multiframe.yaml",
))
BASE_MODEL = os.environ.get("NUREASONING_REASONING_BASE_MODEL", "Qwen/Qwen3.5-4B")
BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")
RUN_TRAINING = os.environ.get("NUREASONING_RUN_REASONING_TRAINING", "0") == "1"
RUN_EVALUATION = os.environ.get("NUREASONING_RUN_REASONING_EVALUATION", "0") == "1"

print("REPO_ROOT", REPO_ROOT)
print("TRAIN_ROOT", TRAIN_ROOT, "exists", TRAIN_ROOT.is_dir())
print("VQA_TRAIN_DIR", VQA_TRAIN_DIR, "exists", VQA_TRAIN_DIR.is_dir())

!python -m nureasoning.vqa.generate \
    --data-root "{TRAIN_ROOT}" --output "{VQA_TRAIN_DIR}"
!python -m nureasoning.vqa.generate \
    --data-root "{VALIDATION_ROOT}" --output "{VQA_VAL_DIR}"

### What a VQA file contains

Each file holds the questions for one frame plus a `temporal_multiview_context` block that indexes
the 8 camera images at each timestamp. Let's look at one.

In [ ]:
import json

# Not every annotated frame yields questions, so take the first one that does.
for vqa_path in sorted(VQA_TRAIN_DIR.rglob("*_vqa.json")):
    doc = json.loads(vqa_path.read_text())
    if doc.get("questions"):
        break
else:
    raise RuntimeError(f"No VQA file with questions found under {VQA_TRAIN_DIR}")

ctx = doc["temporal_multiview_context"]
print(f"clip           {vqa_path.parent.name}")
print(f"frame          {doc['frame_timestamp']}")
print(f"cameras        {len(ctx['camera_sequences'])}")
print(f"frames/camera  {len(next(iter(ctx['camera_sequences'].values())))}  "
      f"(relative indices {ctx['history_layout']})")
print(f"questions      {doc['num_questions']}")

seen = set()
for q in doc["questions"]:
    key = (q["category"], q["question_type"])
    if key in seen:
        continue
    seen.add(key)
    print(f"\n[{q['category']} / {q['question_type']}] {q['subcategory']}")
    print(f"  Q: {q['question'][:150]}")
    if q.get("choices"):
        print(f"  choices: {q['choices']}")
    print(f"  A: {q.get('answer')}  {str(q.get('answer_text') or '')[:80]}")

## 2. Build the supervised fine-tuning dataset

`build_sft` flattens the VQA files into one training example per question, each with its ordered
image list, a prompt, and the target answer.

Four flags decide what the model sees, and the same values must be passed to `evaluate` later:

- `--num-forward-frames 2` — the current frame and the one 1 s earlier, per camera.
- `--history-first` — emit the images time-major (all 8 cameras at t-1s, then all 8 at t) rather
  than camera-major.
- `--keyframe-only` — one frame per clip, so a single clip does not flood the set with
  near-duplicate views.
- `--max-spatial-per-frame 10` — cap the Spatial category, which dominates by count, and keep the
  rarer Decision and Counterfactual questions in full.

In [ ]:
SFT_PATH = REASONING_WORKSPACE / "sft_train_multiframe.jsonl"
!python -m nureasoning.reasoning.build_sft \
    --vqa-dir "{VQA_TRAIN_DIR}" \
    --output "{SFT_PATH}" \
    --keyframe-only --num-forward-frames 2 --max-images 16 --history-first \
    --max-spatial-per-frame 10

### What a training example looks like

The prompt ends with an answer-format instruction taken from the same module the scorer uses, so
the model is trained to answer in exactly the shape evaluation parses.

In [ ]:
from collections import Counter

rows = [json.loads(line) for line in SFT_PATH.read_text().splitlines()]
print(f"{len(rows)} examples from {len({r['clip_dir'] for r in rows})} clips")
print("by question type:", Counter(r["question_type"] for r in rows))
print("by category:     ", Counter(r["category"] for r in rows))

row = rows[0]
print(f"\n{len(row['images'])} images, time-major "
      f"(the last 8 are the current timestamp):")
for i, p in enumerate(row["images"]):
    print(f"  [{i:2}] {Path(p).parent.name:>12} / {Path(p).name}")

print(f"\nPROMPT\n{row['question']}")
print(f"\nTARGET\n{row['assistant']}")

## 3. LoRA fine-tuning

Only the LoRA adapter trains, and the loss is computed on the assistant answer alone; prompt and
image tokens are masked out. The backbone class is read from the checkpoint's own config, so the
same command trains Qwen3.5, Qwen3-VL, or another image-text-to-text model.

The default config writes under `./reasoning_workspace` relative to the repository root (the
working directory after the first cell). The effective batch is `num_gpus x per_device_train_batch_size x
gradient_accumulation_steps`; the reference runs use 64, so raise `gradient_accumulation_steps`
when training on fewer GPUs. Full training is opt-in: set
`NUREASONING_RUN_REASONING_TRAINING=1` before starting the notebook.

In [ ]:
if RUN_TRAINING:
    !python -m nureasoning.reasoning.train \
        --config "{REASONING_CONFIG}"
else:
    print("Skipping full LoRA training. Set NUREASONING_RUN_REASONING_TRAINING=1 to run it.")

On multiple GPUs, run it with `torchrun` instead (from the repository root):

```bash
torchrun --standalone --nproc_per_node=8 -m nureasoning.reasoning.train \
    --config nureasoning/reasoning/configs/qwen3.5-4b-multiframe.yaml
```

Interrupted runs resume with their optimizer and scheduler state via
`--resume-from-checkpoint <output_dir>/checkpoint-<N>`.

## 4. Merge the adapter

Inference servers load a plain checkpoint far faster than base weights plus an adapter. The merge
also drops tied `lm_head` weights and copies auxiliary heads that LoRA never touches, such as
Qwen3.5's multi-token-prediction head.

`train` appends the run date to the output directory, so check the exact name it printed.

In [ ]:
adapters = sorted(
    REASONING_WORKSPACE.glob("lora_qwen3.5-4b-multiframe*"),
    key=lambda p: p.stat().st_mtime,
)
ADAPTER_DIR = adapters[-1] if adapters else None
print(f"Latest adapter: {ADAPTER_DIR}" if ADAPTER_DIR else "No adapter found; run training first.")

In [ ]:
MERGED_DIR = REASONING_WORKSPACE / "merged_qwen3.5-4b-multiframe"
RUN_MERGE = os.environ.get("NUREASONING_RUN_REASONING_MERGE", "0") == "1"
if RUN_MERGE and ADAPTER_DIR:
    !python -m nureasoning.reasoning.merge_lora \
        --base "{BASE_MODEL}" \
        --adapter "{ADAPTER_DIR}" \
        --out "{MERGED_DIR}"
else:
    print("Skipping merge. Set NUREASONING_RUN_REASONING_MERGE=1 after training to run it.")

## 5. Serve the merged model

Evaluation talks to any OpenAI-compatible endpoint. Serve the merged checkpoint **in a separate
terminal** with a **dedicated** `vllm` conda env — do not `pip install vllm` into `nureasoning`,
it overwrites the training PyTorch stack. Setup is in [`docs/installation.md`](../docs/installation.md#reasoning-training--testing).
Match `--tensor-parallel-size` to the number of GPUs allocated to that server:

```bash
conda activate vllm
VLLM_USE_FLASHINFER_SAMPLER=0 vllm serve ./reasoning_workspace/merged_qwen3.5-4b-multiframe \
    --served-model-name nureasoning-4b-sft \
    --tensor-parallel-size 8 \
    --max-model-len 24576 \
    --dtype bfloat16 \
    --mm-processor-kwargs '{"max_pixels": 200704}' \
    --enable-prefix-caching
```

With 16 images per request, startup takes several minutes — multimodal warmup and KV-cache
profiling dominate. `max_pixels` must match the value in the training config.

Set `NUREASONING_CHECK_REASONING_SERVER=1` before running the next cell to wait for the external
server. Set `OPENAI_BASE_URL` when it is not listening at the default local URL.

In [ ]:
import time
import urllib.request

CHECK_SERVER = os.environ.get("NUREASONING_CHECK_REASONING_SERVER", "0") == "1"
if CHECK_SERVER:
    for _ in range(180):
        try:
            with urllib.request.urlopen(f"{BASE_URL}/models", timeout=5) as r:
                names = [m["id"] for m in json.load(r)["data"]]
            print("server ready, serving:", names)
            break
        except Exception:
            time.sleep(10)
    else:
        print("server did not come up; check the vLLM log")
else:
    print("Skipping server wait. Set NUREASONING_CHECK_REASONING_SERVER=1 when vLLM is running.")

## 6. Evaluate

The image flags must match those used in step 2. Questions that share a frame are sent together
and their images are encoded once, so with prefix caching the vision tokens of a frame are
prefilled once for all of its questions. Predictions stream to disk in manifest order, so an
interrupted run still leaves a scoreable `predictions.jsonl`.

Add `--limit 200` for a quick smoke test before committing to the full split.

In [ ]:
EVAL_LIMIT = int(os.environ.get("NUREASONING_REASONING_EVAL_LIMIT", "200"))
if RUN_EVALUATION:
    !python -m nureasoning.reasoning.evaluate \
        --vqa-dir "{VQA_VAL_DIR}" \
        --model nureasoning-4b-sft \
        --base-url "{BASE_URL}" \
        --output-root "{REASONING_EVAL_ROOT}" \
        --keyframe-only --num-forward-frames 2 --max-images 16 --history-first \
        --max-spatial-per-frame 10 --limit {EVAL_LIMIT}
else:
    print("Skipping evaluation. Start an OpenAI-compatible server and set NUREASONING_RUN_REASONING_EVALUATION=1.")

## 7. Read the results

`metrics.yaml` holds every metric aggregated by question type, category, subcategory, and
(question type, category) pair.

In [ ]:
import yaml

METRICS_PATH = REASONING_EVAL_ROOT / "nureasoning-4b-sft" / "metrics_eval" / "metrics.yaml"
if METRICS_PATH.is_file():
    metrics = yaml.safe_load(METRICS_PATH.read_text())["metrics"]

    print("counts:", metrics["counts"])
    print("\nHeadline metric per question type")
    for qtype, value in sorted(metrics["primary_metric_mean_by_question_type"].items()):
        print(f"  {qtype:<12} {100 * value:6.1f}")

    print("\nPer category")
    interesting = [
        "choice_letter_exact",
        "numerical_within_tolerance",
        "coordinate_hit_at_tolerance",
        "trajectory_hit_at_tolerance",
        "text_rougeL_f1",
        "text_token_f1",
        "categorical_accuracy",
    ]
    for category, entries in sorted(metrics["by_category"].items()):
        parts = [f"{k}={100 * entries[k]['mean']:.1f}" for k in interesting if k in entries]
        print(f"  {category:<22} n={max(e['count'] for e in entries.values()):<6} {'  '.join(parts)}")
else:
    print(f"No metrics found at {METRICS_PATH}; run evaluation first.")

## 8. Slice the results by scenario type

Long-tail behaviour is the point of the benchmark, so the interesting question is usually not the
overall number but how it splits across scenario types. Every clip's `metadata.json` carries a
multi-label `scenario_type` field, and every scored row identifies its clip, so the two join
directly.

In [ ]:
import re
from collections import defaultdict

SCENARIO = "work_zone_or_roadwork"   # any tag used in metadata.json's scenario_type
RESULTS_PATH = (
    REASONING_EVAL_ROOT / "nureasoning-4b-sft" / "reasoning_results" /
    "evaluation_vs_groundtruth.jsonl"
)

if RESULTS_PATH.is_file():
    tags_by_clip = {}
    for metadata_path in VALIDATION_ROOT.rglob("metadata.json"):
        scenario_type = str(json.loads(metadata_path.read_text()).get("scenario_type") or "")
        tags_by_clip[metadata_path.parent.name] = {
            t.strip().lower() for t in re.split(r"[|,;]", scenario_type) if t.strip()
        }
    print(f"{len(tags_by_clip)} clips, "
          f"{sum(SCENARIO in t for t in tags_by_clip.values())} tagged '{SCENARIO}'")

    rows = [json.loads(line) for line in RESULTS_PATH.read_text().splitlines()]
    buckets = defaultdict(lambda: defaultdict(list))
    for row in rows:
        if row.get("error") or row.get("primary_metric") is None:
            continue
        clip = row.get("clip_dir") or row["sample_id"].split("/")[0]
        in_scenario = SCENARIO in tags_by_clip.get(clip, set())
        for split in ("overall", SCENARIO if in_scenario else f"non-{SCENARIO}"):
            buckets[split][row["question_type"]].append(row["primary_metric"])

    splits = ("overall", SCENARIO, f"non-{SCENARIO}")
    qtypes = sorted({q for split in buckets.values() for q in split})
    width = max(len(s) for s in splits) + 2
    print(f"{'split':<{width}}" + "".join(f"{q:>14}" for q in qtypes))
    for split in splits:
        scores = buckets.get(split, {})
        cells = "".join(
            f"{100 * sum(scores[q]) / len(scores[q]):>14.1f}" if scores.get(q) else f"{'-':>14}"
            for q in qtypes
        )
        print(f"{split:<{width}}{cells}")
else:
    print(f"No detailed results found at {RESULTS_PATH}; run evaluation first.")

## Next steps

- Swap the backbone by pointing a config at a different checkpoint; no code changes are needed.
- Score an unmodified base model or a hosted API by pointing step 6 at a different endpoint — that
  is how the zero-shot rows of the paper's tables were produced.
- Rescore an existing run after changing a metric with `python -m nureasoning.reasoning.reaggregate`.
- Submit reasoning answers together with planning for the challenge: see
  [`docs/submission.md`](../docs/submission.md).